这是一个MLP

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words=open('names.txt','r').read().splitlines()


In [3]:
#依旧遍历然后建立词表stoi,itos
chars=sorted(set(''.join(words)))#先建立一个里面有的字符的排序表chars
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}


# 开始构建主体

先梳理清楚，接下来开始构建的是MLP的主体，前向传播部分。MLP的组成原理从逻辑上来说，就是一次看三个字符，这就是我们要达到的效果，比方说abcd里面根据前面三个来预测第四个是啥。之前我们使用的一个的。然后因为用三个如果还用之前那种概率表形式的话参数会太多了，你想27*27*27可能性太多了，更别说后面还要乘以输入，表大的你算的绝望。所以改用emb的方式，即先把某个词嵌入2维向量，比方说a我就用(1,2)来表示（具体用什么表示是模型自己训练出来的，我们一开始初始化一个随机的就行。）然后再把这个玩意和参数W相乘。得到的输出再softmax一下，最后输出一个向量，即27维的，每个维度对应那个维度的概率是多少。然后损失函数就根据xs和ys的错位，也就是ys是一个字母，xs是三个字母嘛，我们依旧用之前的似然值作为损失函数，也就是把当时那前面几个xs的输出预测的那个ys给softmax，再log，然后来求平均概率的，再加一个正则化就行

## 一些对我难点：
首先拼接维度那里，我要用到的一些函数我还不是很熟悉。比如cat之类的，torch的功能实在是太多了，还有一些维度上面的规定我也不是很熟悉。后面的部分和之前的很相似我觉得应该还好，就是前期的处理我需要注意一下。以及把看的词块写灵活用的block_size等，先写着看吧。

In [7]:
block_size=3
X,Y=[],[]
for w in words[:5]:
    print(w)
    context=[0]*block_size
    for ch in w+'.':
        
        #接下来要做的是：先打印一次context，这是我们前三个观测的值，放到输入xs里
        # 然后可以打印其下一个值，并把下一个词放到ys里。再之后移动context即可
        ix=stoi[ch]
        Y.append(ix)
        X.append(context)
        print(''.join(itos[i] for i in context),'--->',itos[ix])
        context=context[1:]+[ix]
X=torch.tensor(X)
Y=torch.tensor(Y)
X
        

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [8]:
C=torch.randn((27,2))

#处理C[X]和W的矛盾有两种办法
#用unbind➕cat，一个拆一个拼
#直接view，最轮椅，pytorch里最方便的。有点类似数据结构里面
#那个线性结构和储存方式，即存在内存里其实就一维，只是读取方式不同
emd=C[X]
#所以接下来就是改一下规模，因为我们想要他们能成只能转化为矩阵
W1=torch.randn(6,100)
b1=torch.randn(100)
#torch.cat(torch.unbind(emd,1),1).shape
emd.view(32,6)





tensor([[-0.1582, -0.1724, -0.1582, -0.1724, -0.1582, -0.1724],
        [-0.1582, -0.1724, -0.1582, -0.1724, -0.2626,  0.5382],
        [-0.1582, -0.1724, -0.2626,  0.5382,  0.5246, -0.0902],
        [-0.2626,  0.5382,  0.5246, -0.0902,  0.5246, -0.0902],
        [ 0.5246, -0.0902,  0.5246, -0.0902, -1.5418, -0.4262],
        [-0.1582, -0.1724, -0.1582, -0.1724, -0.1582, -0.1724],
        [-0.1582, -0.1724, -0.1582, -0.1724, -0.9021, -0.0676],
        [-0.1582, -0.1724, -0.9021, -0.0676,  1.2904,  1.1350],
        [-0.9021, -0.0676,  1.2904,  1.1350,  0.5236,  1.0078],
        [ 1.2904,  1.1350,  0.5236,  1.0078,  2.1872, -1.7463],
        [ 0.5236,  1.0078,  2.1872, -1.7463,  0.5236,  1.0078],
        [ 2.1872, -1.7463,  0.5236,  1.0078, -1.5418, -0.4262],
        [-0.1582, -0.1724, -0.1582, -0.1724, -0.1582, -0.1724],
        [-0.1582, -0.1724, -0.1582, -0.1724, -1.5418, -0.4262],
        [-0.1582, -0.1724, -1.5418, -0.4262,  2.1872, -1.7463],
        [-1.5418, -0.4262,  2.1872, -1.7